In [ ]:
!pip install -q langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import re
import pandas as pd
import os
import io

from google.colab import auth
from googleapiclient.discovery import build
from google.auth import default
from googleapiclient.http import MediaIoBaseDownload

FILE_ID = "1ZfMR_y92P6stVVupl1QGykcknl6EJU8X"
pdf_path = "/content/npr-7150-2d.pdf"

if not os.path.exists(pdf_path):
    auth.authenticate_user()

    creds, _ = default()
    drive = build("drive", "v3", credentials=creds)

    request = drive.files().get_media(
        fileId=FILE_ID,
        supportsAllDrives=True
    )

    with io.FileIO(pdf_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)

        done = False
        while not done:
            status, done = downloader.next_chunk()
            print(f"Downloaded {int(status.progress() * 100)}%")

    print(f"PDF downloaded to: {pdf_path}")

else:
    print(f"PDF already exists at: {pdf_path}")

Downloaded 100%
PDF downloaded to: /content/npr-7150-2d.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")

/tmp/ipykernel_1055/3021515166.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 89 pages


In [ ]:
full_text = ""
page_ranges = []

for page_num, page in enumerate(pages, start=1):
    text = page.page_content

    text = re.sub(
        r"NPR 7150\.2D -- [^\n]+\n"
        r"This document does not bind the public.*?"
        r"Page\s+\d+\s+of\s+\d+",
        "",
        text,
        flags=re.DOTALL
    )

    text = re.sub(
        r"NPR 7150\.2D -- [^\n]+ Page\s+\d+\s+of\s+\d+",
        "",
        text
    )

    start = len(full_text)

    full_text += text.strip() + "\n\n"

    end = len(full_text)

    page_ranges.append({
        "page": page_num,
        "start": start,
        "end": end
    })

print(full_text[:2000])

| NODIS Library | Program Formulation(7000s) | Search | 
 NASA
Procedural
Requirements 
NPR 7150.2D 
Effective Date: March 08, 2022
Expiration Date: March 08, 2027
COMPLIANCE IS MANDATORY FOR NASA EMPLOYEES 
NASA Software Engineering Requirements
Responsible Office: Office of the Chief Engineer
Table of Contents 
Preface
P.1 Purpose
P.2 Applicability
P.3 Authority
P.4 Applicable Documents and Forms
P.5 Measurement/Verification
P.6 Cancellation 
Chapter 1. Introduction
1.1 Overview
1.2 Hierarchy of NASA Software-Related Engineering and Program/Project Documents
1.3 Document Structure 
Chapter 2. Roles, Responsibilities, and Principles Related to
Tailoring of the Requirements 
2.1 Roles and Responsibilities 
2.2 Principles Related to Tailoring of the Requirements 
Chapter 3. Software Management Requirements
3.1 Software Life Cycle Planning
3.2 Software Cost Estimation
3.3 Software Schedules
3.4 Software Training

3.4 Software Training
3.5 Software Classification Assessments
3.6 Software 

In [ ]:
section_pattern = re.compile(
    r"(?m)^(\d+\.\d+(?:\.\d+)*)\s+"
)

section_matches = list(section_pattern.finditer(full_text))

print(f"Found {len(section_matches)} numbered sections")

Found 380 numbered sections


In [ ]:
def get_page_number(offset):
    for page_info in page_ranges:
        if page_info["start"] <= offset < page_info["end"]:
            return page_info["page"]

    return None

In [ ]:
pd.set_option("display.max_colwidth", 200)

In [ ]:
rows = []

for i, match in enumerate(section_matches):
    section = match.group(1)

    start = match.end()

    if i + 1 < len(section_matches):
        end = section_matches[i + 1].start()
    else:
        end = len(full_text)

    block_text = full_text[start:end].strip()

    swe_match = re.search(r"\[SWE-(\d+)\]", block_text)

    if not swe_match:
        continue

    swe_id = f"SWE-{swe_match.group(1)}"

    requirement_text = re.sub(
        r"\[SWE-\d+\]",
        "",
        block_text
    ).strip()

    requirement_text = re.split(
        r"\nChapter\s+\d+[:.]",
        requirement_text
    )[0].strip()

    page = get_page_number(match.start())

    rows.append({
        "swe_id": swe_id,
        "section": section,
        "requirement_text": requirement_text,
        "page": page
    })

requirements_df = pd.DataFrame(rows)

print(f"Found {len(requirements_df)} SWE requirements")

requirements_df

Found 130 SWE requirements


,swe_id,section,requirement_text,page
0,SWE-002,2.1.1.1,The NASA OCE shall lead and maintain a NASA Software Engineering Initiative to advance\nsoftware engineering practices.,11
1,SWE-004,2.1.1.2,The NASA OCE shall periodically benchmark each Center’s software engineering capability\nagainst requirements in this directive. \nNote: Capability Maturity Model® Integration (CMMI®) for Develop...,11
2,SWE-152,2.1.1.3,The NASA OCE shall periodically review the project requirements mapping matrices.,11
3,SWE-129,2.1.1.4,The NASA OCE shall authorize appraisals against selected requirements in this NPR to\ncheck compliance.,11
4,SWE-100,2.1.1.5,The NASA OCE and Center training organizations shall provide training to advance\nsoftware engineering practices.,11
...,...,...,...,...
125,SWE-200,5.4.6,"The project manager shall collect, track, and report software requirements volatility metrics.",42
126,SWE-201,5.5.1,The project manager shall track and maintain software non-conformances (including defects in\ntools and appropriate ground software).,42
127,SWE-202,5.5.2,"The project manager shall define and implement clear software severity levels for all software\nnon-conformances (including tools, COTS, GOTS, MOTS, OSS, reused software components, and\napplicabl...",42
128,SWE-203,5.5.3,"The project manager shall implement mandatory assessments of reported non-conformances\nfor all COTS, GOTS, MOTS, OSS, and/or reused software components. \n\nNote: This includes operating systems,...",42


In [ ]:
chapters_3_to_5 = requirements_df[
    requirements_df["section"].str.match(r"^[3-5]\.")
]

chapters_3_to_5

,swe_id,section,requirement_text,page
30,SWE-033,3.1.2,"The project manager shall assess options for software acquisition versus development.\n \nNote: The assessment can include risk, cost, and benefits criteria for each of the options\nlisted below: ...",20
31,SWE-013,3.1.3,"The project manager shall develop, maintain, and execute software plans, including security\nplans, that cover the entire software life cycle and, as a minimum, address the requirements of this\nd...",20
32,SWE-024,3.1.4,"The project manager shall track the actual results and performance of software activities\nagainst the software plans. \na. Corrective actions are taken, recorded, and managed to closure. \nb. Ch...",21
33,SWE-034,3.1.5,The project manager shall define and document the acceptance criteria for the software.,21
34,SWE-036,3.1.6,"The project manager shall establish and maintain the software processes, software\ndocumentation plans, list of developed electronic products, deliverables, and list of tasks for the\nsoftware dev...",21
...,...,...,...,...
125,SWE-200,5.4.6,"The project manager shall collect, track, and report software requirements volatility metrics.",42
126,SWE-201,5.5.1,The project manager shall track and maintain software non-conformances (including defects in\ntools and appropriate ground software).,42
127,SWE-202,5.5.2,"The project manager shall define and implement clear software severity levels for all software\nnon-conformances (including tools, COTS, GOTS, MOTS, OSS, reused software components, and\napplicabl...",42
128,SWE-203,5.5.3,"The project manager shall implement mandatory assessments of reported non-conformances\nfor all COTS, GOTS, MOTS, OSS, and/or reused software components. \n\nNote: This includes operating systems,...",42


In [ ]:
requirements_df["chapter"] = (
    requirements_df["section"]
    .str.split(".")
    .str[0]
)

requirements_df["chapter"].value_counts().sort_index()

,count
chapter,
2,29
3,45
4,34
5,21
7150,1


In [ ]:
requirements_df[
    requirements_df["section"].str.startswith("7150")
]

,swe_id,section,requirement_text,page,chapter
28,SWE-150,7150.2,requirements per the requirements mapping matrix authority column.,19,7150


In [ ]:
requirements_df = requirements_df[
    ~requirements_df["section"].str.startswith("7150", na=False)
]
requirements_df

,swe_id,section,requirement_text,page,chapter
0,SWE-002,2.1.1.1,The NASA OCE shall lead and maintain a NASA Software Engineering Initiative to advance\nsoftware engineering practices.,11,2
1,SWE-004,2.1.1.2,The NASA OCE shall periodically benchmark each Center’s software engineering capability\nagainst requirements in this directive. \nNote: Capability Maturity Model® Integration (CMMI®) for Develop...,11,2
2,SWE-152,2.1.1.3,The NASA OCE shall periodically review the project requirements mapping matrices.,11,2
3,SWE-129,2.1.1.4,The NASA OCE shall authorize appraisals against selected requirements in this NPR to\ncheck compliance.,11,2
4,SWE-100,2.1.1.5,The NASA OCE and Center training organizations shall provide training to advance\nsoftware engineering practices.,11,2
...,...,...,...,...,...
125,SWE-200,5.4.6,"The project manager shall collect, track, and report software requirements volatility metrics.",42,5
126,SWE-201,5.5.1,The project manager shall track and maintain software non-conformances (including defects in\ntools and appropriate ground software).,42,5
127,SWE-202,5.5.2,"The project manager shall define and implement clear software severity levels for all software\nnon-conformances (including tools, COTS, GOTS, MOTS, OSS, reused software components, and\napplicabl...",42,5
128,SWE-203,5.5.3,"The project manager shall implement mandatory assessments of reported non-conformances\nfor all COTS, GOTS, MOTS, OSS, and/or reused software components. \n\nNote: This includes operating systems,...",42,5
